# Classical Abel–Jacobi MNIST training

MNIST models are defined **in this notebook** (CNN trunk + heads). The **`aj`** library provides `AJGridActivationNorm` (forward grid lookup) and `InverseAbelJacobiNetwork` (inverse map).

- **Forward (g=30):** CNN → per-point heads → `AJGridActivationNorm` → `Linear(2g, 10)`.
- **Inverse (g=30):** CNN → `Linear(64, 2g)` → `InverseAbelJacobiNetwork` (branch points from the same tables; toy Ω/K) → `Linear(2g, 10)` on **u**.


## Config

Run from repo root or `notebooks/`.

In [1]:
import sys
from pathlib import Path

for p in [Path.cwd(), Path.cwd().parent]:
    if str(p) not in sys.path:
        sys.path.insert(0, str(p))

from src import util as aj_util

REPO_ROOT = aj_util.get_repo_root(Path.cwd())
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

DATA_ROOT = REPO_ROOT / "data"
TABLES_DIR = ""
FORWARD_CKPT = ""
INVERSE_CKPT = ""
TEST_SUBSET = 0


## MNIST model definitions

Shared CNN trunk and full models. Only `AJGridActivationNorm` / `InverseAbelJacobiNetwork` come from **`aj.classical`**.

In [2]:
import numpy as np
import torch
import torch.nn as nn

from aj.classical import InverseAbelJacobiNetwork
from aj.classical.grid_activation import AJGridActivationNorm


def mnist_cnn_trunk() -> nn.Sequential:
    """32×32×1 MNIST (after pad) → 1×1×64 feature map."""
    return nn.Sequential(
        nn.Conv2d(1, 32, 3, padding=1),
        nn.ReLU(),
        nn.MaxPool2d(2),
        nn.Conv2d(32, 64, 3, padding=1),
        nn.ReLU(),
        nn.MaxPool2d(2),
        nn.AdaptiveAvgPool2d((1, 1)),
    )


class AJMNIST_Anchored(nn.Module):
    """
    CNN trunk → g learned point heads → AJGridActivationNorm → Linear(2g, 10).
    CrossEntropyLoss applies log-softmax on logits.
    """

    def __init__(
        self,
        genus,
        I_plus,
        Om_plus,
        grid_r,
        grid_i,
        branch_pts,
        anchors_xy,
        mu,
        sigma,
        embed_dim=8,
    ):
        super().__init__()
        self.genus = genus
        self.conv = mnist_cnn_trunk()
        self.embed = nn.Parameter(torch.empty(genus, embed_dim))
        nn.init.uniform_(self.embed, -2.0, 2.0)
        self.point_head = nn.Linear(64 + embed_dim, 3, bias=False)
        nn.init.xavier_uniform_(self.point_head.weight)
        self.point_bias = nn.Parameter(torch.zeros(genus, 3))
        self.aj = AJGridActivationNorm(I_plus, Om_plus, grid_r, grid_i, branch_pts, mu, sigma)
        self.classifier = nn.Linear(2 * genus, 10)
        rmin, rmax = float(grid_r.min()), float(grid_r.max())
        imin, imax = float(grid_i.min()), float(grid_i.max())

        def logit(p):
            p = np.clip(p, 1e-6, 1 - 1e-6)
            return float(np.log(p / (1 - p)))

        with torch.no_grad():
            for i in range(genus):
                x0, y0 = float(anchors_xy[i, 0]), float(anchors_xy[i, 1])
                px = (x0 - rmin) / (rmax - rmin)
                py = (y0 - imin) / (imax - imin)
                self.point_bias[i, 0] = logit(px)
                self.point_bias[i, 1] = logit(py)
                target_sign = 0.8 if (i % 2 == 0) else -0.8
                self.point_bias[i, 2] = float(torch.atanh(torch.tensor(target_sign)))

    def forward(self, x, return_aux=False):
        B = x.size(0)
        h = self.conv(x).view(B, -1)
        h_exp = h.unsqueeze(1).expand(-1, self.genus, -1)
        emb = self.embed.unsqueeze(0).expand(B, -1, -1)
        out = self.point_head(torch.cat([h_exp, emb], dim=2)) + self.point_bias.unsqueeze(0)
        raw_xy, sheet_logits = out[..., :2], out[..., 2]
        coords, aux = self.aj(raw_xy, sheet_logits, return_aux=True)
        logits = self.classifier(coords)
        if return_aux:
            return logits, aux
        return logits


class TorusFeatures(nn.Module):
    def __init__(self, dim: int, K: int = 2, freqs=None, learnable: bool = False):
        super().__init__()
        if freqs is None:
            freqs = torch.tensor([0.5, 1.0], dtype=torch.float32)[:K]
        else:
            freqs = torch.as_tensor(freqs, dtype=torch.float32)[:K]
        if learnable:
            self.freqs = nn.Parameter(freqs)
        else:
            self.register_buffer("freqs", freqs)
        self.dim, self.K = dim, len(freqs)

    def forward(self, u):
        B, D = u.shape
        f = self.freqs.view(1, 1, -1).to(u.device)
        ang = u.unsqueeze(-1) * f
        return torch.cat([torch.cos(ang), torch.sin(ang)], dim=-1).view(B, D * 2 * self.K)


class AJMNIST_AxisPeriodic(nn.Module):
    """Anchored AJ coords → torus features → Linear(2g·2K, 10)."""

    def __init__(
        self,
        genus,
        I_plus,
        Om_plus,
        grid_r,
        grid_i,
        branch_pts,
        anchors_xy,
        mu,
        sigma,
        embed_dim=8,
        K=2,
        learnable_freqs=False,
    ):
        super().__init__()
        self.base = AJMNIST_Anchored(
            genus,
            I_plus,
            Om_plus,
            grid_r,
            grid_i,
            branch_pts,
            anchors_xy,
            mu,
            sigma,
            embed_dim=embed_dim,
        )
        D = 2 * genus
        self.torus = TorusFeatures(D, K=K, learnable=learnable_freqs)
        self.classifier = nn.Linear(D * 2 * K, 10)

    def forward(self, x, return_aux=False):
        B = x.size(0)
        h = self.base.conv(x).view(B, -1)
        h_exp = h.unsqueeze(1).expand(-1, self.base.genus, -1)
        emb = self.base.embed.unsqueeze(0).expand(B, -1, -1)
        out = self.base.point_head(torch.cat([h_exp, emb], dim=2)) + self.base.point_bias.unsqueeze(0)
        raw_xy, sheet_logits = out[..., :2], out[..., 2]
        coords, aux = self.base.aj(raw_xy, sheet_logits, return_aux=True)
        feats = self.torus(coords)
        logits = self.classifier(feats)
        if return_aux:
            return logits, aux
        return logits


def branch_points_tensor_from_tables(tables: dict) -> torch.Tensor:
    """(2g+2, 2) real branch coordinates from loaded forward tables."""
    bp = tables["branch_pts_t"]
    if torch.is_complex(bp):
        return torch.stack([bp.real.float(), bp.imag.float()], dim=1)
    bp = bp.detach().cpu().numpy()
    if np.iscomplexobj(bp):
        return torch.tensor(np.stack([bp.real, bp.imag], axis=1), dtype=torch.float32)
    return torch.tensor(bp, dtype=torch.float32)


def toy_period_matrix(genus: int) -> tuple[np.ndarray, np.ndarray]:
    """Placeholder Ω, K for large genus (skip mpmath period quadrature)."""
    Omega = (1.0 + 2.0j) * np.eye(genus, dtype=np.complex128)
    Omega += 0.1 * (np.ones((genus, genus)) - np.eye(genus))
    Omega = (Omega + Omega.T) / 2
    K = np.zeros(genus, dtype=np.complex128)
    return Omega, K


def _toy_log_sigma(v: np.ndarray) -> complex:
    v = np.asarray(v, dtype=np.complex128).ravel()
    if v.size == 1:
        return -(v[0] ** 3) / 6.0
    return -(v[0] ** 3) / 6.0 - 0.5 * v[0] * (v[1] ** 2)


INVERSE_MAX_GENUS_FOR_PERIOD_COMPUTE = 4


def build_inverse_aj_net(
    tables: dict,
    device,
    genus: int,
    base_point=(-8.0, -8.0),
) -> InverseAbelJacobiNetwork:
    """InverseAbelJacobiNetwork on the same branch locus as the forward tables."""
    branch_points = branch_points_tensor_from_tables(tables)
    if genus <= INVERSE_MAX_GENUS_FOR_PERIOD_COMPUTE:
        Omega_init, K_init = None, None
    else:
        Omega_init, K_init = toy_period_matrix(genus)
        print(
            f"Inverse AJ: genus {genus} > {INVERSE_MAX_GENUS_FOR_PERIOD_COMPUTE}; "
            "using toy Ω, K (skip mpmath period computation)."
        )
    return InverseAbelJacobiNetwork(
        genus=genus,
        branch_points=branch_points,
        base_point=base_point,
        init_divisor_points=torch.zeros(genus, 2, dtype=torch.float32),
        use_kleinian_p=True,
        log_sigma_fun=_toy_log_sigma,
        Omega_init=Omega_init,
        K_init=K_init,
    ).to(device)


class MNISTInversePNet(nn.Module):
    """CNN → u ∈ R^{B×g×2} → InverseAbelJacobiNetwork → Linear(2g, 10) on u."""

    def __init__(self, inv_net: InverseAbelJacobiNetwork):
        super().__init__()
        self.genus = inv_net.genus
        self.conv = mnist_cnn_trunk()
        self.to_u = nn.Linear(64, 2 * self.genus)
        self.inv_net = inv_net
        self.classifier = nn.Linear(2 * self.genus, 10)

    def forward(self, x):
        h = self.conv(x).view(x.size(0), -1)
        u = self.to_u(h).view(-1, self.genus, 2)
        _ = self.inv_net(u)
        return self.classifier(u.reshape(x.size(0), 2 * self.genus))


## Train forward AJ

Loads genus-30 tables via `aj_util.get_or_build_forward_tables`. **Default matches `AJ_training_genus30.ipynb`:** `AJMNIST_AxisPeriodic` (K=2), two-tier AdamW, AMP on CUDA. Set `FORWARD_AXIS_PERIODIC = False` for the smaller anchored head only.

In [3]:
import torch
import torch.nn as nn
from types import SimpleNamespace

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device for forward AJ training:", device)

aj_test = SimpleNamespace(
    _safe_torch_load=aj_util.safe_torch_load,
    ensure_mnist_available=aj_util.ensure_mnist_available,
    load_forward_tables=aj_util.load_forward_tables,
    eval_epoch=aj_util.eval_epoch,
)

FORWARD_EPOCHS = 20
FORWARD_LR_BASE = 3e-4   # conv + AJGridActivationNorm (genus30 notebook "base")
FORWARD_LR_FAST = 1e-3   # point_head, point_bias, classifier (+ torus if axis-periodic)
FORWARD_WEIGHT_DECAY = 1e-4
FORWARD_TRAIN_SUBSET = 20000
FORWARD_BATCH_SIZE = 64
FORWARD_TEST_BATCH_SIZE = 256
FORWARD_CKPT_OUT = REPO_ROOT / "checkpoints" / "aj_forward_mnist_genus30.pt"
FORWARD_AXIS_PERIODIC = True   # genus30 trains axis-periodic, not anchored-only
FORWARD_K = 2
FORWARD_LEARN_FREQS = False
FORWARD_EMBED_DIM = 4
REFRESH_FORWARD_NORM = False   # genus30 uses mu/sigma from tables (mean/std on grid), not nan refresh

aj_util.ensure_mnist_available(str(DATA_ROOT))
train_loader_full, test_loader_fwd = aj_util.get_mnist_loaders(
    root=str(DATA_ROOT),
    test_batch_size=FORWARD_TEST_BATCH_SIZE,
    num_workers=0,
)
train_ds = train_loader_full.dataset
if FORWARD_TRAIN_SUBSET > 0:
    n = min(FORWARD_TRAIN_SUBSET, len(train_ds))
    train_ds = torch.utils.data.Subset(train_ds, list(range(n)))
    print(f"Forward AJ: using train subset of {n} samples")

train_loader_fwd = torch.utils.data.DataLoader(
    train_ds, batch_size=FORWARD_BATCH_SIZE, shuffle=True, num_workers=0,
)

TABLES_GENUS = 30
tables, tables_found_dir = aj_util.get_or_build_forward_tables(
    device=device,
    tables_dir=(TABLES_DIR if TABLES_DIR else None),
    data_root=DATA_ROOT,
    genus=TABLES_GENUS,
    auto_build=True,
    grid_size=96,
)
print(f"Using forward AJ tables from: {tables_found_dir}")

if REFRESH_FORWARD_NORM:
    aj_util.refresh_forward_table_normalization(tables, device=device)

common_kw = dict(
    genus=tables["genus"],
    I_plus=tables["I_plus"],
    Om_plus=tables["Om_plus"],
    grid_r=tables["grid_r"],
    grid_i=tables["grid_i"],
    branch_pts=tables["branch_pts_t"],
    anchors_xy=tables["anchors_xy_t"],
    mu=tables["mu_t"],
    sigma=tables["sigma_t"],
    embed_dim=FORWARD_EMBED_DIM,
)
if FORWARD_AXIS_PERIODIC:
    model_fwd = AJMNIST_AxisPeriodic(
        **common_kw, K=FORWARD_K, learnable_freqs=FORWARD_LEARN_FREQS
    ).to(device)
else:
    model_fwd = AJMNIST_Anchored(**common_kw).to(device)

opt_fwd = aj_util.forward_aj_adamw(
    model_fwd,
    lr_base=FORWARD_LR_BASE,
    lr_fast=FORWARD_LR_FAST,
    weight_decay=FORWARD_WEIGHT_DECAY,
)
USE_FORWARD_AMP = device.type == "cuda"
fwd_scaler = torch.cuda.amp.GradScaler(enabled=USE_FORWARD_AMP)
LAM_BRANCH = 1e-3
LAM_BOUND = 1e-3
CLIP_NORM = 1.0

metrics_history = {
    "forward": {"epoch": [], "train_loss": [], "train_acc": [], "test_loss": [], "test_acc": []},
    "inverse": {"epoch": [], "train_loss": [], "train_acc": [], "test_loss": [], "test_acc": []},
}

print("Starting forward AJ training (AMP =", USE_FORWARD_AMP, ")...")
for ep in range(1, FORWARD_EPOCHS + 1):
    tr_loss, tr_acc = aj_util.train_forward_aj_epoch_amp(
        model_fwd,
        train_loader_fwd,
        opt_fwd,
        device,
        clip=CLIP_NORM,
        lam_branch=LAM_BRANCH,
        lam_bound=LAM_BOUND,
        scaler=fwd_scaler,
        use_amp=USE_FORWARD_AMP,
    )
    te_loss, te_acc = aj_util.eval_epoch(model_fwd, test_loader_fwd, device)
    metrics_history["forward"]["epoch"].append(ep)
    metrics_history["forward"]["train_loss"].append(tr_loss)
    metrics_history["forward"]["train_acc"].append(tr_acc)
    metrics_history["forward"]["test_loss"].append(te_loss)
    metrics_history["forward"]["test_acc"].append(te_acc)
    print(f"[Forward AJ] Epoch {ep:02d} | train {tr_loss:.4f}/{tr_acc:.2f}% | test {te_loss:.4f}/{te_acc:.2f}%")

FORWARD_CKPT_OUT.parent.mkdir(parents=True, exist_ok=True)
torch.save({"state_dict": model_fwd.state_dict()}, FORWARD_CKPT_OUT)
FORWARD_CKPT = str(FORWARD_CKPT_OUT)
print("Saved forward AJ checkpoint to", FORWARD_CKPT)


Using device for forward AJ training: cuda
Forward AJ: using train subset of 20000 samples
Using forward AJ tables from: /home/users/hshunt/Abel-Jacobi-Networks/data/AJ_Tables_g30
Starting forward AJ training (AMP = True )...


/tmp/ipykernel_63899/1559152467.py:85: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  fwd_scaler = torch.cuda.amp.GradScaler(enabled=USE_FORWARD_AMP)
/home/users/hshunt/Abel-Jacobi-Networks/src/util.py:732: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=use_amp):


[Forward AJ] Epoch 01 | train 2.3127/10.50% | test 2.3122/11.35%
[Forward AJ] Epoch 02 | train 2.3114/10.36% | test 2.3210/11.35%
[Forward AJ] Epoch 03 | train 2.3114/10.71% | test 2.3112/11.35%
[Forward AJ] Epoch 04 | train 2.3135/10.35% | test 2.3141/11.35%
[Forward AJ] Epoch 05 | train 2.3115/10.54% | test 2.3155/10.32%
[Forward AJ] Epoch 06 | train 2.3125/10.35% | test 2.3126/11.35%
[Forward AJ] Epoch 07 | train 2.3103/10.29% | test 2.3129/9.58%
[Forward AJ] Epoch 08 | train 2.3115/10.52% | test 2.3088/9.74%
[Forward AJ] Epoch 09 | train 2.3119/10.74% | test 2.3056/11.35%
[Forward AJ] Epoch 10 | train 2.3099/10.36% | test 2.3023/10.32%
[Forward AJ] Epoch 11 | train 2.3116/10.26% | test 2.3216/10.10%
[Forward AJ] Epoch 12 | train 2.3129/10.61% | test 2.3143/11.35%
[Forward AJ] Epoch 13 | train 2.3121/10.47% | test 2.3121/9.58%
[Forward AJ] Epoch 14 | train 2.3112/10.08% | test 2.3130/11.35%
[Forward AJ] Epoch 15 | train 2.3110/10.36% | test 2.3163/9.74%
[Forward AJ] Epoch 16 | train

## Train inverse AJ (genus 30)

In [4]:
import torch
import torch.nn as nn
import torchvision
import torchvision.transforms as T

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device for inverse AJ training:", device)

INVERSE_GENUS = 30
INVERSE_EPOCHS = 20
INVERSE_BATCH_SIZE = 64
INVERSE_TRAIN_SUBSET = 20000
INVERSE_TEST_SUBSET = 0
INVERSE_CKPT_OUT = REPO_ROOT / "checkpoints" / f"mnist_inverse_p_g{INVERSE_GENUS}.pt"

# Reuse genus-30 forward tables for branch locus (same as forward section if already loaded).
if "tables" not in globals() or int(tables.get("genus", -1)) != INVERSE_GENUS:
    tables, tables_found_dir = aj_util.get_or_build_forward_tables(
        device=device,
        tables_dir=(TABLES_DIR if TABLES_DIR else None),
        data_root=DATA_ROOT,
        genus=INVERSE_GENUS,
        auto_build=True,
        grid_size=96,
    )
    print(f"Inverse AJ: loaded tables from {tables_found_dir}")

inv_core = build_inverse_aj_net(tables, device, genus=INVERSE_GENUS)
model_inv = MNISTInversePNet(inv_core).to(device)

tfm = T.Compose([T.ToTensor(), T.Normalize((0.1307,), (0.3081,))])
train_ds_inv = torchvision.datasets.MNIST(root=str(DATA_ROOT), train=True, download=True, transform=tfm)
test_ds_inv = torchvision.datasets.MNIST(root=str(DATA_ROOT), train=False, download=True, transform=tfm)

if INVERSE_TRAIN_SUBSET > 0:
    n = min(INVERSE_TRAIN_SUBSET, len(train_ds_inv))
    train_ds_inv = torch.utils.data.Subset(train_ds_inv, list(range(n)))
    print(f"Inverse AJ: using train subset of {n} samples")

if INVERSE_TEST_SUBSET > 0:
    m = min(INVERSE_TEST_SUBSET, len(test_ds_inv))
    test_ds_inv = torch.utils.data.Subset(test_ds_inv, list(range(m)))

train_loader_inv = torch.utils.data.DataLoader(
    train_ds_inv, batch_size=INVERSE_BATCH_SIZE, shuffle=True, num_workers=0,
)
test_loader_inv = torch.utils.data.DataLoader(
    test_ds_inv, batch_size=max(INVERSE_BATCH_SIZE, 128), shuffle=False, num_workers=0,
)

opt_inv = torch.optim.AdamW(model_inv.parameters(), lr=3e-4, weight_decay=1e-4)
ce = nn.CrossEntropyLoss()


def train_epoch_inv(model, loader, opt, device):
    model.train()
    tot, correct, n = 0.0, 0, 0
    for x, y in loader:
        x, y = x.to(device), y.to(device)
        opt.zero_grad(set_to_none=True)
        logits = model(x)
        loss = ce(logits, y)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        opt.step()
        tot += loss.item() * x.size(0)
        correct += (logits.argmax(1) == y).sum().item()
        n += x.size(0)
    return tot / n, 100.0 * correct / n


if "metrics_history" not in globals():
    metrics_history = {
        "forward": {"epoch": [], "train_loss": [], "train_acc": [], "test_loss": [], "test_acc": []},
        "inverse": {"epoch": [], "train_loss": [], "train_acc": [], "test_loss": [], "test_acc": []},
    }

print(f"Training MNIST inverse P model: genus={INVERSE_GENUS}, epochs={INVERSE_EPOCHS}")
for ep in range(1, INVERSE_EPOCHS + 1):
    tr_loss, tr_acc = train_epoch_inv(model_inv, train_loader_inv, opt_inv, device)
    te_loss, te_acc = aj_util.eval_epoch(model_inv, test_loader_inv, device)
    metrics_history["inverse"]["epoch"].append(ep)
    metrics_history["inverse"]["train_loss"].append(tr_loss)
    metrics_history["inverse"]["train_acc"].append(tr_acc)
    metrics_history["inverse"]["test_loss"].append(te_loss)
    metrics_history["inverse"]["test_acc"].append(te_acc)
    print(f"[Inverse AJ g={INVERSE_GENUS}] Epoch {ep:02d} | train {tr_loss:.4f}/{tr_acc:.2f}% | test {te_loss:.4f}/{te_acc:.2f}%")

INVERSE_CKPT_OUT.parent.mkdir(parents=True, exist_ok=True)
torch.save({"state_dict": model_inv.state_dict()}, INVERSE_CKPT_OUT)
INVERSE_CKPT = str(INVERSE_CKPT_OUT)
print("Saved inverse AJ checkpoint to", INVERSE_CKPT)


Using device for inverse AJ training: cuda
Inverse AJ: genus 30 > 4; using toy Ω, K (skip mpmath period computation).
Inverse AJ: using train subset of 20000 samples
Training MNIST inverse P model: genus=30, epochs=20


KeyboardInterrupt: 

## Training curves (forward vs inverse)

Requires `metrics_history` from the forward and inverse training cells above.

In [ ]:
import matplotlib.pyplot as plt

if "metrics_history" not in globals():
    raise RuntimeError("Run the forward and inverse training cells first to populate metrics_history.")

fwd = metrics_history["forward"]
inv = metrics_history["inverse"]

fig, (ax_acc, ax_loss) = plt.subplots(1, 2, figsize=(12, 4.5))

if fwd["epoch"]:
    ax_acc.plot(fwd["epoch"], fwd["train_acc"], "o-", label="Forward train", color="C0")
    ax_acc.plot(fwd["epoch"], fwd["test_acc"], "s--", label="Forward test", color="C0", alpha=0.85)
    ax_loss.plot(fwd["epoch"], fwd["train_loss"], "o-", label="Forward train", color="C0")
    ax_loss.plot(fwd["epoch"], fwd["test_loss"], "s--", label="Forward test", color="C0", alpha=0.85)

if inv["epoch"]:
    ax_acc.plot(inv["epoch"], inv["train_acc"], "o-", label="Inverse train", color="C1")
    ax_acc.plot(inv["epoch"], inv["test_acc"], "s--", label="Inverse test", color="C1", alpha=0.85)
    ax_loss.plot(inv["epoch"], inv["train_loss"], "o-", label="Inverse train", color="C1")
    ax_loss.plot(inv["epoch"], inv["test_loss"], "s--", label="Inverse test", color="C1", alpha=0.85)

ax_acc.set_xlabel("Epoch")
ax_acc.set_ylabel("Accuracy (%)")
ax_acc.set_title("Train / test accuracy")
ax_acc.legend(loc="best", fontsize=9)
ax_acc.grid(True, alpha=0.3)

ax_loss.set_xlabel("Epoch")
ax_loss.set_ylabel("Cross-entropy loss")
ax_loss.set_title("Train / test loss")
ax_loss.legend(loc="best", fontsize=9)
ax_loss.grid(True, alpha=0.3)

fig.suptitle("MNIST: forward AJ (g=30) vs inverse AJ (g=30)", y=1.02)
plt.tight_layout()
plt.show()

# Summary table (last epoch)
def _last(h, key):
    return h[key][-1] if h[key] else float("nan")

print(
    f"{'':12} {'train acc':>12} {'test acc':>12} {'train loss':>12} {'test loss':>12}"
)
for name, h in [("Forward", fwd), ("Inverse", inv)]:
    if not h["epoch"]:
        continue
    print(
        f"{name:12} {_last(h, 'train_acc'):11.2f}% {_last(h, 'test_acc'):11.2f}% "
        f"{_last(h, 'train_loss'):12.4f} {_last(h, 'test_loss'):12.4f}"
    )